In [0]:
WITH bronze_stats AS (
    SELECT COUNT(*) AS rows
    FROM workspace.retail_dev_schema.bronze_transactions
),
silver_stats AS (
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT invoice) AS orders,
        COUNT(DISTINCT stock_code) AS products,
        COUNT(DISTINCT CASE
            WHEN customer_id IS NOT NULL
             AND TRIM(customer_id) <> ''
            THEN customer_id
        END) AS customers
    FROM workspace.retail_dev_schema.silver_order_items
),
daily_groups AS (
    SELECT COUNT(*) AS rows
    FROM (
        SELECT source_date, country
        FROM workspace.retail_dev_schema.silver_order_items
        GROUP BY source_date, country
    )
),
counts AS (
    SELECT
        'Bronze' AS layer,
        'bronze_transactions' AS table_name,
        COUNT(*) AS actual_rows,
        CAST(NULL AS BIGINT) AS expected_rows
    FROM workspace.retail_dev_schema.bronze_transactions

    UNION ALL

    SELECT
        'Silver', 'silver_order_items',
        COUNT(*), (SELECT rows FROM bronze_stats)
    FROM workspace.retail_dev_schema.silver_order_items

    UNION ALL

    SELECT
        'Silver', 'silver_orders',
        COUNT(*), (SELECT orders FROM silver_stats)
    FROM workspace.retail_dev_schema.silver_orders

    UNION ALL

    SELECT
        'Silver', 'silver_products',
        COUNT(*), (SELECT products FROM silver_stats)
    FROM workspace.retail_dev_schema.silver_products

    UNION ALL

    SELECT
        'Silver', 'silver_customers_current',
        COUNT(*), (SELECT customers FROM silver_stats)
    FROM workspace.retail_dev_schema.silver_customers_current

    UNION ALL

    SELECT
        'Silver', 'silver_customers_history',
        COUNT(*), CAST(NULL AS BIGINT)
    FROM workspace.retail_dev_schema.silver_customers_history

    UNION ALL

    SELECT
        'Gold', 'gold_daily_sales',
        COUNT(*), (SELECT rows FROM daily_groups)
    FROM workspace.retail_dev_schema.gold_daily_sales

    UNION ALL

    SELECT
        'Gold', 'gold_product_sales',
        COUNT(*), (SELECT products FROM silver_stats)
    FROM workspace.retail_dev_schema.gold_product_sales

    UNION ALL

    SELECT
        'Check', 'Gold: represented transaction lines',
        COALESCE(SUM(line_count), 0),
        (SELECT rows FROM silver_stats)
    FROM workspace.retail_dev_schema.gold_daily_sales

    UNION ALL

    SELECT
        'Check', 'SCD2: current customer versions',
        COUNT(*), (SELECT customers FROM silver_stats)
    FROM workspace.retail_dev_schema.silver_customers_history
    WHERE is_current = true
)
SELECT
    layer,
    table_name,
    actual_rows,
    expected_rows,
    CASE
        WHEN expected_rows IS NULL THEN 'INFO'
        WHEN actual_rows = expected_rows THEN 'PASS'
        ELSE 'FAIL'
    END AS count_check
FROM counts
ORDER BY
    CASE layer
        WHEN 'Bronze' THEN 1
        WHEN 'Silver' THEN 2
        WHEN 'Gold' THEN 3
        ELSE 4
    END,
    table_name;